# BeeSpace 03 - CLMS Land Cover

Simulação de uso e cobertura do solo no entorno das colmeias. Em operação, esta camada deve ser substituída por produtos do Copernicus Land Monitoring Service.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(exist_ok=True)

In [ ]:
classes = ["mata_nativa", "agricultura", "solo_exposto", "agua", "area_urbana"]
colmeias = pd.DataFrame({"id_colmeia": ["C001", "C002", "C003", "C004", "C005"]})

valores = np.random.dirichlet(alpha=[3, 3, 1, 0.5, 1], size=len(colmeias))
landcover = pd.DataFrame(valores, columns=classes)
landcover = pd.concat([colmeias, landcover], axis=1)

for c in classes:
    landcover[c] = (landcover[c] * 100).round(1)

landcover

In [ ]:
def calcular_qualidade_entorno(row):
    return (
        row["mata_nativa"] * 1.2
        + row["agua"] * 0.2
        - row["agricultura"] * 0.4
        - row["solo_exposto"] * 1.0
        - row["area_urbana"] * 0.5
    )

landcover["score_entorno"] = landcover.apply(calcular_qualidade_entorno, axis=1)
landcover["status_entorno"] = pd.cut(
    landcover["score_entorno"],
    bins=[-100, 15, 45, 100],
    labels=["critico", "atencao", "favoravel"]
)
landcover

In [ ]:
plot_df = landcover.set_index("id_colmeia")[classes]
ax = plot_df.plot(kind="bar", stacked=True, figsize=(9, 5))
plt.title("Uso e cobertura do solo no raio de 3 km")
plt.xlabel("Colmeia")
plt.ylabel("Percentual")
plt.legend(title="Classe", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "landcover_colmeias_beespace.png", dpi=150)
plt.show()

landcover.to_csv(OUTPUT_DIR / "landcover_stats_beespace.csv", index=False)